# MLP、Dropout、多视角与任务专属融合补充实验

沿用70万训练记录及已保存的共同五折，比较MLP有无Dropout。选出最佳单模型配置后，补充三视角等权与任务专属融合消融。融合权重只在外层训练部分的内层OOF预测上学习。五折阶段报告AUC和时间，最终选定后确定两个OOF阈值并评价补充数据。

## 数据、环境与运行身份

In [ ]:
from copy import deepcopy
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo
import fcntl
import gc
import hashlib
import importlib.metadata
import json
import os
import threading
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import psutil
import torch
from torch import nn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from scipy.optimize import minimize
from threadpoolctl import threadpool_limits
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

PROJECT_ROOT = Path(os.environ.get("ML_PROJECT_ROOT", ".")).resolve()
config = json.loads((PROJECT_ROOT / "configs/course_extended_run_20261112.json").read_text())
PARENT = PROJECT_ROOT / "outputs/course_study" / config["parent_run_id"]
RUN = PROJECT_ROOT / "outputs/course_study" / config["run_id"]
for folder in ["checkpoints", "predictions", "evaluation"]:
    (RUN / folder).mkdir(parents=True, exist_ok=True)
run_lock = (RUN / "execution.lock").open("a+")
fcntl.flock(run_lock.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
parent_configuration = json.loads((PARENT / "configuration.json").read_text())
contract = parent_configuration["features"]
BASE = contract["allowed_columns"]
CATEGORICAL = contract["categorical_columns"]
VIEWS = contract["groups"]
targets = parent_configuration["config"]["targets"]
parameters = parent_configuration["parameters"]
release = parent_configuration["release"]
seed = 2026
threadpool_limits(limits=4)
torch.set_num_threads(4)
torch.set_num_interop_threads(1)
torch.use_deterministic_algorithms(True)


def atomic_json(path, value):
    path = Path(path)
    temporary = path.with_name("." + path.name + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    temporary.replace(path)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def hash_value(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str).encode()).hexdigest()


def atomic_joblib(path, value):
    path = Path(path)
    temporary = path.with_name("." + path.name + ".tmp")
    joblib.dump(value, temporary, compress=3)
    temporary.replace(path)


def atomic_torch(path, value):
    path = Path(path)
    temporary = path.with_name("." + path.name + ".tmp")
    torch.save(value, temporary)
    temporary.replace(path)


fit_records = json.loads((RUN / "fit_records.json").read_text()) if (RUN / "fit_records.json").exists() else {}
attempts = json.loads((RUN / "fit_attempts.json").read_text()) if (RUN / "fit_attempts.json").exists() else []
for attempt in attempts:
    if attempt["status"] == "running":
        attempt["status"] = "interrupted_before_restart"


def progress(phase, **details):
    atomic_json(RUN / "progress.json", {
        "status": "running", "phase": phase, "completed_new_classifier_fits": len(fit_records),
        "planned_cv_and_inner_fits": 140, "maximum_final_fits": 6,
        "updated_at": datetime.now(ZoneInfo("Asia/Shanghai")).isoformat(), **details,
    })


manifest_path = PARENT / "development_folds.parquet"
fold_frame = pd.read_parquet(manifest_path)
IDS = fold_frame["encounter_id"].astype(str).to_numpy()
Y = fold_frame[targets].to_numpy(dtype=np.int8)
fold_ids = fold_frame["fold"].to_numpy(dtype=np.int8)
assert len(IDS) == 700000 and np.isin(Y, [0, 1]).all()
processed = Path(release["processed_directory"])
for name in ["development.parquet", "temporal_test_locked.parquet"]:
    assert sha256_file(processed / name) == release["artifact_sha256"][name]
parent_frozen_check = json.loads((PARENT / "frozen_protocol.json").read_text())
assert sha256_file(PARENT / "cv_summary.csv") == parent_frozen_check["cv_summary_sha256"]
assert sha256_file(PARENT / "predictions/development_oof.parquet") == parent_frozen_check["oof_sha256"]
raw = pd.concat([
    pd.read_parquet(processed / name, columns=["encounter_id"] + BASE + targets)
    for name in ["development.parquet", "temporal_test_locked.parquet"]
], ignore_index=True)
raw["encounter_id"] = raw.encounter_id.astype(str)
raw = raw.set_index("encounter_id").loc[IDS].reset_index()
np.testing.assert_array_equal(raw[targets].to_numpy(dtype=np.int8), Y)
values = {}
for column in BASE:
    if column in CATEGORICAL:
        values[column] = raw[column].astype("string").fillna("__MISSING__").astype(object)
    else:
        values[column] = pd.to_numeric(raw[column], errors="raise").to_numpy(dtype=np.float32, na_value=np.nan)
X = pd.DataFrame(values)
del raw, values
gc.collect()
versions = {name: importlib.metadata.version(name) for name in ["numpy", "pandas", "scipy", "scikit-learn", "catboost", "lightgbm", "xgboost", "torch", "pyarrow"]}
run_signature = hash_value({"config": config, "contract": contract, "parameters": parameters, "versions": versions,
                            "fold_manifest_sha256": sha256_file(manifest_path), "data_release": release["source_bundle_sha256"]})
if (RUN / "configuration.json").exists():
    assert json.loads((RUN / "configuration.json").read_text())["signature"] == run_signature
else:
    atomic_json(RUN / "configuration.json", {"signature": run_signature, "config": config, "features": contract,
                                            "parameters": parameters, "versions": versions, "release": release})
fold_frame.to_parquet(RUN / "development_folds.parquet", index=False)
print("Extended experiment", config["run_id"], "Torch", versions["torch"], flush=True)


## 复用已核验的预处理与阈值实现

In [ ]:
def encode_inputs(fitting, prediction_frames, columns, algorithm):
    categorical = [column for column in CATEGORICAL if column in columns]
    numeric = [column for column in columns if column not in categorical]
    if algorithm in {"LightGBM", "CatBoost"}:
        frequent = {}
        levels = {}
        for column in categorical:
            counts = fitting[column].value_counts()
            frequent[column] = set(counts[counts >= 50].index)
            levels[column] = sorted(frequent[column] | {"__MISSING__", "__OTHER__"})
        state = {
            "kind":"native", "algorithm":algorithm, "columns":columns,
            "categorical":categorical, "frequent":frequent, "levels":levels,
        }
        def transform(frame):
            result = frame[columns].copy()
            for column in categorical:
                result[column] = result[column].where(
                    result[column].isin(frequent[column]), "__OTHER__"
                )
                if algorithm == "LightGBM":
                    result[column] = pd.Categorical(
                        result[column], categories=levels[column]
                    )
            return result
        fitting_matrix = transform(fitting)
        prediction_matrices = {
            key: transform(frame) for key, frame in prediction_frames.items()
        }
    else:
        steps = [SimpleImputer(strategy="median", keep_empty_features=True)]
        if algorithm == "LogisticRegression":
            steps.append(StandardScaler())
        encoder = ColumnTransformer([
            ("numeric", make_pipeline(*steps), numeric),
            ("categorical", OneHotEncoder(
                min_frequency=50, handle_unknown="infrequent_if_exist",
                sparse_output=True, dtype=np.float32,
            ), categorical),
        ], sparse_threshold=1.0)
        fitting_matrix = encoder.fit_transform(fitting[columns]).astype(np.float32)
        prediction_matrices = {
            key: encoder.transform(frame[columns]).astype(np.float32)
            for key, frame in prediction_frames.items()
        }
        state = {
            "kind":"one_hot", "algorithm":algorithm,
            "columns":columns, "categorical":categorical, "encoder":encoder,
        }
    return fitting_matrix, prediction_matrices, state

def transform_saved_preprocessing(frame,state):
    columns = state["columns"]
    if state["kind"] == "one_hot":
        return state["encoder"].transform(frame[columns]).astype(np.float32)
    result = frame[columns].copy()
    for column in state["categorical"]:
        result[column] = result[column].where(
            result[column].isin(state["frequent"][column]),"__OTHER__"
        )
        if state["algorithm"] == "LightGBM":
            result[column] = pd.Categorical(
                result[column],categories=state["levels"][column]
            )
    return result

class PeakMemory:
    def __init__(self):
        self.process = psutil.Process()
        self.peak = self.process.memory_info().rss
        self.finished = threading.Event()
        self.worker = threading.Thread(target=self.sample, daemon=True)

    def sample(self):
        while not self.finished.wait(0.05):
            self.peak = max(self.peak, self.process.memory_info().rss)

    def __enter__(self):
        self.worker.start()
        return self

    def __exit__(self, exception_type, exception_value, traceback):
        self.peak = max(self.peak, self.process.memory_info().rss)
        self.finished.set()
        self.worker.join()

def prepare_base_features(raw):
    values={}
    for column in BASE:
        if column in CATEGORICAL:
            values[column]=raw[column].astype("string").fillna("__MISSING__").astype(object)
        else:
            values[column]=pd.to_numeric(raw[column],errors="raise").to_numpy(dtype=np.float32,na_value=np.nan)
    return pd.DataFrame(values,index=raw.index).reset_index(drop=True)

def youden_threshold(labels,probability):
    labels=np.asarray(labels,dtype=np.int64)
    candidates=np.unique(np.r_[0.0,probability,1.0])
    order=np.argsort(probability,kind="stable")
    sorted_scores=probability[order]
    cumulative=np.r_[0,np.cumsum(labels[order],dtype=np.int64)]
    left=np.searchsorted(sorted_scores,candidates,side="left")
    positive=int(labels.sum())
    negative=len(labels)-positive
    if positive==0 or negative==0:
        raise ValueError("Youden阈值需要两个类别")
    tp=positive-cumulative[left]
    fp=len(labels)-left-tp
    numerators=tp*negative-fp*positive
    winner=np.flatnonzero(numerators==numerators.max())[0]
    return {"threshold":float(candidates[winner]),"youden":float(numerators[winner]/(positive*negative)),"positive_records":positive,"negative_records":negative,"selection":"pooled selected-algorithm OOF; lowest threshold on exact J ties"}

## 神经网络、检查点及融合权重实现

In [ ]:
class BinaryMLP(nn.Module):
    def __init__(self, input_features, dropout):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_features, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, 1),
        )

    def forward(self, inputs):
        return self.layers(inputs).squeeze(1)


def is_neural(method):
    return method in ["MLP", "MLP_Dropout"]


def dense_neural(matrix):
    result = matrix.toarray() if hasattr(matrix, "toarray") else matrix
    result = np.asarray(result, dtype=np.float32, order="C")
    if not np.isfinite(result).all():
        raise ValueError("Neural input contains a nonfinite value after preprocessing")
    return result


def load_classifier(record):
    if record["format"] == "torch_state":
        saved = torch.load(record["model_path"], map_location="cpu", weights_only=True)
        model = BinaryMLP(saved["input_features"], saved["dropout"])
        model.load_state_dict(saved["state_dict"])
        model.eval()
        return model
    return joblib.load(record["model_path"])


def fit_neural(matrix, labels, dropout, directory, target_index, context):
    torch.manual_seed(seed)
    model = BinaryMLP(matrix.shape[1], dropout)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.0001)
    loss_function = nn.BCEWithLogitsLoss()
    generator = torch.Generator().manual_seed(seed)
    inputs = torch.from_numpy(matrix)
    output = torch.from_numpy(labels.astype(np.float32))
    checkpoint = directory / f"target_{target_index}_training.pt"
    history_path = directory / f"target_{target_index}_epochs.json"
    history = []
    first_epoch = 0
    if checkpoint.exists():
        saved = torch.load(checkpoint, map_location="cpu", weights_only=True)
        model.load_state_dict(saved["model"])
        optimizer.load_state_dict(saved["optimizer"])
        generator.set_state(saved["shuffle_rng"])
        torch.set_rng_state(saved["torch_rng"])
        first_epoch = saved["completed_epochs"]
        history = saved["history"]
    for epoch in range(first_epoch, config["neural"]["epochs"]):
        model.train()
        started = time.perf_counter()
        order = torch.randperm(len(inputs), generator=generator)
        loss_total = 0.0
        for beginning in range(0, len(inputs), config["neural"]["batch_size"]):
            indices = order[beginning:beginning + config["neural"]["batch_size"]]
            optimizer.zero_grad(set_to_none=True)
            loss = loss_function(model(inputs[indices]), output[indices])
            if not torch.isfinite(loss):
                raise ValueError("Nonfinite neural training loss")
            loss.backward()
            optimizer.step()
            loss_total += float(loss.detach()) * len(indices)
        seconds = time.perf_counter() - started
        history.append({"epoch": epoch + 1, "training_loss": loss_total / len(inputs), "training_seconds": seconds})
        atomic_torch(checkpoint, {"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                                 "shuffle_rng": generator.get_state(), "torch_rng": torch.get_rng_state(),
                                 "completed_epochs": epoch + 1, "history": history})
        atomic_json(history_path, history)
        progress("neural_training", context=context, target=targets[target_index], epoch=epoch + 1,
                 total_epochs=config["neural"]["epochs"], epoch_seconds=seconds)
    model.eval()
    return model, sum(row["training_seconds"] for row in history)


def fit_pair(context, method, fitting_positions, columns):
    directory = RUN / "checkpoints" / context
    directory.mkdir(exist_ok=True)
    signature = hash_value({"run": run_signature, "context": context, "method": method,
                            "positions": hashlib.sha256(np.asarray(fitting_positions, dtype=np.int64).tobytes()).hexdigest(),
                            "columns": columns})
    preprocessing_path = directory / "preprocessing.joblib"
    preprocessing_metadata_path = directory / "preprocessing.json"
    if preprocessing_metadata_path.exists():
        preprocessing = json.loads(preprocessing_metadata_path.read_text())
        assert preprocessing["signature"] == signature and sha256_file(preprocessing_path) == preprocessing["sha256"]
        state = joblib.load(preprocessing_path)
        fitting = transform_saved_preprocessing(X.iloc[fitting_positions], state)
        if is_neural(method):
            fitting = dense_neural(fitting)
    else:
        started = time.perf_counter()
        fitting, _, state = encode_inputs(X.iloc[fitting_positions], {}, columns,
                                         "LogisticRegression" if is_neural(method) else method)
        if is_neural(method):
            fitting = dense_neural(fitting)
        preprocessing_seconds = time.perf_counter() - started
        atomic_joblib(preprocessing_path, state)
        preprocessing = {"signature": signature, "sha256": sha256_file(preprocessing_path),
                         "preprocessing_seconds": preprocessing_seconds}
        atomic_json(preprocessing_metadata_path, preprocessing)
    statistics, models = [], []
    constructors = {"LogisticRegression": LogisticRegression, "RandomForest": RandomForestClassifier,
                    "XGBoost": XGBClassifier, "LightGBM": LGBMClassifier, "CatBoost": CatBoostClassifier}
    for target_index, target in enumerate(targets):
        key = context + "__" + str(target_index)
        path = directory / (f"target_{target_index}.pt" if is_neural(method) else f"target_{target_index}.joblib")
        metadata_path = directory / f"target_{target_index}.json"
        if metadata_path.exists():
            record = json.loads(metadata_path.read_text())
            assert record["signature"] == signature and sha256_file(path) == record["model_sha256"]
            model = load_classifier(record)
        else:
            progress("fitting", context=context, method=method, target=target)
            attempt = {"context": context, "method": method, "target": target, "status": "running",
                       "started_at": datetime.now(ZoneInfo("Asia/Shanghai")).isoformat()}
            attempts.append(attempt)
            atomic_json(RUN / "fit_attempts.json", attempts)
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")
                with PeakMemory() as memory:
                    if is_neural(method):
                        dropout = 0.2 if method == "MLP_Dropout" else 0.0
                        model, fitting_seconds = fit_neural(fitting, Y[fitting_positions, target_index], dropout, directory, target_index, context)
                    else:
                        model = constructors[method](**deepcopy(parameters[method + "__" + target]))
                        started = time.perf_counter()
                        arguments = {"cat_features": [name for name in CATEGORICAL if name in columns]} if method == "CatBoost" else {}
                        model.fit(fitting, Y[fitting_positions, target_index], **arguments)
                        fitting_seconds = time.perf_counter() - started
            if is_neural(method):
                atomic_torch(path, {"input_features": fitting.shape[1], "dropout": dropout, "state_dict": model.state_dict()})
            else:
                atomic_joblib(path, model)
            record = {"signature": signature, "context": context, "method": method, "target": target,
                      "fitting_rows": len(fitting_positions), "fitting_seconds": fitting_seconds,
                      "peak_fitting_process_rss_mib": memory.peak / (1024 ** 2),
                      "warnings": [str(item.message) for item in caught], "model_path": str(path),
                      "model_sha256": sha256_file(path), "format": "torch_state" if is_neural(method) else "joblib"}
            atomic_json(metadata_path, record)
            attempt.update(status="completed", fitting_seconds=fitting_seconds)
            atomic_json(RUN / "fit_attempts.json", attempts)
        fit_records[key] = record
        atomic_json(RUN / "fit_records.json", fit_records)
        statistics.append(record)
        models.append(model)
    del fitting
    gc.collect()
    return {"directory": directory, "signature": signature, "method": method, "state": state,
            "preprocessing": preprocessing, "statistics": statistics, "models": models,
            "training_seconds": preprocessing["preprocessing_seconds"] + sum(row["fitting_seconds"] for row in statistics)}


def predict_pair(pair, frame, partition, identifiers):
    path = pair["directory"] / (partition + "_probabilities.npz")
    metadata_path = pair["directory"] / (partition + "_prediction.json")
    signature = hash_value({"pair": pair["signature"], "model_hashes": [row["model_sha256"] for row in pair["statistics"]],
                            "ids": hash_value(list(map(str, identifiers))), "partition": partition})
    if metadata_path.exists():
        statistics = json.loads(metadata_path.read_text())
        assert statistics["signature"] == signature and sha256_file(path) == statistics["sha256"]
        with np.load(path, allow_pickle=False) as archive:
            probability = archive["probability"]
        return probability, statistics
    started = time.perf_counter()
    matrix = transform_saved_preprocessing(frame, pair["state"])
    if is_neural(pair["method"]):
        matrix = torch.from_numpy(dense_neural(matrix))
    probability = np.empty((len(frame), 2), dtype=float)
    for target_index, model in enumerate(pair["models"]):
        if is_neural(pair["method"]):
            model.eval()
            with torch.inference_mode():
                for beginning in range(0, len(matrix), 8192):
                    probability[beginning:beginning + 8192, target_index] = torch.sigmoid(model(matrix[beginning:beginning + 8192])).numpy()
        else:
            probability[:, target_index] = model.predict_proba(matrix)[:, 1]
    seconds = time.perf_counter() - started
    assert np.isfinite(probability).all() and ((probability >= 0) & (probability <= 1)).all()
    temporary = path.with_name("." + path.stem + ".tmp.npz")
    np.savez_compressed(temporary, probability=probability)
    temporary.replace(path)
    statistics = {"signature": signature, "prediction_seconds": seconds,
                  "seconds_per_10000": seconds * 10000 / len(frame), "sha256": sha256_file(path)}
    atomic_json(metadata_path, statistics)
    return probability, statistics


def choose_weights(predictions, labels):
    started = time.perf_counter()
    epsilon = 1e-7

    def objective(weights):
        probability = np.clip(predictions @ weights, epsilon, 1 - epsilon)
        loss = -np.mean(labels * np.log(probability) + (1 - labels) * np.log1p(-probability))
        gradient = predictions.T @ ((probability - labels) / (probability * (1 - probability))) / len(labels)
        return loss, gradient

    result = minimize(objective, np.full(3, 1 / 3), jac=True, method="SLSQP", bounds=[(0, 1)] * 3,
                      constraints={"type": "eq", "fun": lambda w: w.sum() - 1, "jac": lambda w: np.ones(3)},
                      options={"ftol": 1e-10, "maxiter": 300})
    if not result.success:
        raise RuntimeError("Fusion weight optimization failed: " + result.message)
    weights = np.clip(result.x, 0, 1)
    weights /= weights.sum()
    return weights, time.perf_counter() - started


def metrics_row(fold, method, probability, positions, training_seconds, prediction_seconds):
    auc = [roc_auc_score(Y[positions, index], probability[:, index]) for index in range(2)]
    result = {"fold": fold, "method": method, "training_rows": 700000 - len(positions), "validation_rows": len(positions),
              "auc_A": auc[0], "auc_R": auc[1], "macro_auc": float(np.mean(auc)),
              "training_seconds": training_seconds, "prediction_seconds": prediction_seconds,
              "prediction_seconds_per_10000": prediction_seconds * 10000 / len(positions)}
    previous = saved_cv_records.get((method, fold))
    if previous is not None:
        np.testing.assert_allclose([previous[name] for name in ["auc_A", "auc_R", "macro_auc"]],
                                   [result[name] for name in ["auc_A", "auc_R", "macro_auc"]], atol=1e-12, rtol=0)
        for name in ["training_seconds", "prediction_seconds", "prediction_seconds_per_10000"]:
            result[name] = previous[name]
    return result


## 共同五折与两个模块消融

In [ ]:
saved_cv_records = {}
if (RUN / "cv_fold_metrics.csv").exists():
    for record in pd.read_csv(RUN / "cv_fold_metrics.csv").to_dict(orient="records"):
        saved_cv_records[(record["method"], record["fold"])] = record
parent_oof = pd.read_parquet(PARENT / "predictions/development_oof.parquet")
assert parent_oof.encounter_id.astype(str).tolist() == list(IDS)
parent_metrics = pd.read_csv(PARENT / "cv_fold_metrics.csv")
columns = ["fold", "method", "training_rows", "validation_rows", "auc_A", "auc_R", "macro_auc",
           "training_seconds", "prediction_seconds", "prediction_seconds_per_10000"]
cv_rows = parent_metrics[columns].to_dict(orient="records")
oof = {method: parent_oof[[method + "__" + target for target in targets]].to_numpy()
       for method in parent_metrics.method.unique()}
for method in ["MLP", "MLP_Dropout"]:
    oof[method] = np.full((len(X), 2), np.nan)
for fold in range(5):
    fitting_positions = np.flatnonzero(fold_ids != fold)
    validation_positions = np.flatnonzero(fold_ids == fold)
    for method in ["MLP", "MLP_Dropout"]:
        pair = fit_pair(f"fold{fold}_{method}", method, fitting_positions, BASE)
        probability, statistics = predict_pair(pair, X.iloc[validation_positions], "validation", IDS[validation_positions])
        oof[method][validation_positions] = probability
        cv_rows.append(metrics_row(fold, method, probability, validation_positions,
                                   pair["training_seconds"], statistics["prediction_seconds"]))
        pd.DataFrame(cv_rows).to_csv(RUN / "cv_fold_metrics.csv", index=False)
        print("Neural CV completed", fold, method, cv_rows[-1]["macro_auc"], flush=True)
        del pair, probability
        gc.collect()


def summarize_cv(rows):
    frame = pd.DataFrame(rows)
    return frame.groupby("method", as_index=False).agg(
        auc_A_mean=("auc_A", "mean"), auc_A_sd=("auc_A", "std"),
        auc_R_mean=("auc_R", "mean"), auc_R_sd=("auc_R", "std"),
        macro_auc_mean=("macro_auc", "mean"), macro_auc_sd=("macro_auc", "std"),
        training_seconds_mean=("training_seconds", "mean"), training_seconds_sd=("training_seconds", "std"),
        prediction_seconds_per_10000_mean=("prediction_seconds_per_10000", "mean"),
        prediction_seconds_per_10000_sd=("prediction_seconds_per_10000", "std"),
    ).sort_values(["macro_auc_mean", "training_seconds_mean"], ascending=[False, True])


single_summary = summarize_cv(cv_rows)
single_summary.to_csv(RUN / "single_model_cv_summary.csv", index=False)
best_single = single_summary.iloc[0]["method"]
atomic_json(RUN / "selected_single_model.json", {"method": best_single, "selection": "common five-fold mean Macro AUC", "rows": 700000})
print("Selected single-model configuration", best_single, flush=True)
view_names = list(VIEWS)
assert len(view_names) == 3 and sum(len(VIEWS[name]) for name in view_names) == len(BASE)
oof_views = np.full((len(X), 2, 3), np.nan)
oof["MultiviewEqual"] = np.full((len(X), 2), np.nan)
oof["TaskSpecificFusion"] = np.full((len(X), 2), np.nan)
fold_weight_records = []
for fold in range(5):
    fitting_positions = np.flatnonzero(fold_ids != fold)
    validation_positions = np.flatnonzero(fold_ids == fold)
    inner_probability = np.full((len(fitting_positions), 2, 3), np.nan)
    inner_training_seconds = 0.0
    strata = Y[fitting_positions, 0] * 2 + Y[fitting_positions, 1]
    inner_splits = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed).split(np.zeros(len(strata)), strata)
    for inner_fold, (inner_fitting, inner_validation) in enumerate(inner_splits):
        global_fitting = fitting_positions[inner_fitting]
        global_validation = fitting_positions[inner_validation]
        for view_index, view in enumerate(view_names):
            context = f"fold{fold}_inner{inner_fold}_{view}_{best_single}"
            pair = fit_pair(context, best_single, global_fitting, VIEWS[view])
            probability, statistics = predict_pair(pair, X.iloc[global_validation], "inner_validation", IDS[global_validation])
            inner_probability[inner_validation, :, view_index] = probability
            inner_training_seconds += pair["training_seconds"]
            del pair, probability
            gc.collect()
    assert np.isfinite(inner_probability).all()
    weights_path = RUN / f"fold{fold}_fusion_weights.json"
    weights_signature = hash_value({"run": run_signature, "fold": fold, "backbone": best_single,
                                    "inner_probability": hashlib.sha256(inner_probability.tobytes()).hexdigest()})
    if weights_path.exists():
        weight_record = json.loads(weights_path.read_text())
        assert weight_record["signature"] == weights_signature
        weights = np.array(weight_record["weights"])
        optimization_seconds = weight_record["optimization_seconds"]
    else:
        weights = np.empty((2, 3))
        optimization_seconds = 0.0
        for target_index in range(2):
            weights[target_index], seconds = choose_weights(inner_probability[:, target_index, :], Y[fitting_positions, target_index])
            optimization_seconds += seconds
        atomic_json(weights_path, {"signature": weights_signature, "weights": weights.tolist(),
                                   "optimization_seconds": optimization_seconds, "view_order": view_names})
    for target_index, target in enumerate(targets):
        fold_weight_records.append({"fold": fold, "target": target, **dict(zip(view_names, map(float, weights[target_index])))})
    outer_training_seconds = 0.0
    outer_prediction_seconds = 0.0
    for view_index, view in enumerate(view_names):
        pair = fit_pair(f"fold{fold}_outer_{view}_{best_single}", best_single, fitting_positions, VIEWS[view])
        probability, statistics = predict_pair(pair, X.iloc[validation_positions], "validation", IDS[validation_positions])
        oof_views[validation_positions, :, view_index] = probability
        outer_training_seconds += pair["training_seconds"]
        outer_prediction_seconds += statistics["prediction_seconds"]
        del pair, probability
        gc.collect()
    for method in ["MultiviewEqual", "TaskSpecificFusion"]:
        started = time.perf_counter()
        if method == "MultiviewEqual":
            probability = oof_views[validation_positions].mean(axis=2)
            training_seconds = outer_training_seconds
        else:
            probability = np.einsum("ntv,tv->nt", oof_views[validation_positions], weights)
            training_seconds = inner_training_seconds + outer_training_seconds + optimization_seconds
        prediction_seconds = outer_prediction_seconds + time.perf_counter() - started
        oof[method][validation_positions] = probability
        cv_rows.append(metrics_row(fold, method, probability, validation_positions, training_seconds, prediction_seconds))
        print("Module CV completed", fold, method, cv_rows[-1]["macro_auc"], flush=True)
    pd.DataFrame(cv_rows).to_csv(RUN / "cv_fold_metrics.csv", index=False)
    pd.DataFrame(fold_weight_records).to_csv(RUN / "fold_fusion_weights.csv", index=False)
    del inner_probability
    gc.collect()

assert sum(not record["context"].startswith("final_") for record in fit_records.values()) == 140
assert all(np.isfinite(value).all() for value in oof.values())
summary = summarize_cv(cv_rows)
summary.to_csv(RUN / "cv_summary.csv", index=False)
selected = summary.iloc[0]["method"]
atomic_json(RUN / "selected_algorithm.json", {"method": selected, "backbone": best_single,
                                            "selection": "mean Macro AUC over five common folds; exact tie then mean training time",
                                            "selected_before_new_holdout_evaluation": True})
output_oof = fold_frame.copy()
for method, probability in oof.items():
    for target_index, target in enumerate(targets):
        output_oof[method + "__" + target] = probability[:, target_index]
output_oof.to_parquet(RUN / "predictions/development_oof.parquet", index=False)
for view_index, view in enumerate(view_names):
    for target_index, target in enumerate(targets):
        output_oof[view + "__" + target] = oof_views[:, target_index, view_index]
output_oof[["encounter_id", "fold"] + [view + "__" + target for view in view_names for target in targets]].to_parquet(RUN / "predictions/view_oof.parquet", index=False)
nn_ablation = summary[summary.method.isin(["MLP", "MLP_Dropout"])]
nn_ablation.to_csv(RUN / "neural_ablation_summary.csv", index=False)
summary[summary.method.isin([best_single, "MultiviewEqual", "TaskSpecificFusion"])].to_csv(RUN / "module_ablation_summary.csv", index=False)
pd.DataFrame(cv_rows).query("method in ['MLP', 'MLP_Dropout']").to_csv(RUN / "neural_ablation_folds.csv", index=False)
pd.DataFrame(cv_rows).loc[lambda frame: frame.method.isin([best_single, "MultiviewEqual", "TaskSpecificFusion"])].to_csv(RUN / "module_ablation_folds.csv", index=False)
thresholds = {target: youden_threshold(Y[:, index], oof[selected][:, index]) for index, target in enumerate(targets)}
atomic_json(RUN / "youden_thresholds.json", thresholds)
print("Selected final configuration", selected, "backbone", best_single, flush=True)
progress("final_fitting", selected=selected, best_single=best_single)


## 最终拟合、冻结、评价与提交文件

In [ ]:
import shutil

parent_frozen = json.loads((PARENT / "frozen_protocol.json").read_text())
reuse_parent = selected == parent_frozen["selected_algorithm"]
final_pairs = {}
final_weights = np.full((2, 3), 1 / 3)
final_refit_seconds = 0.0
if reuse_parent:
    for target in targets:
        assert thresholds[target]["threshold"] == parent_frozen["thresholds"][target]["threshold"]
    final_recipe = {"kind": "parent_frozen_model", "source_run": config["parent_run_id"],
                    "source_frozen_protocol_sha256": sha256_file(PARENT / "frozen_protocol.json"),
                    "classifiers": parent_frozen["classifiers"], "preprocessing_path": parent_frozen["preprocessing_path"],
                    "preprocessing_sha256": parent_frozen["preprocessing_sha256"]}
elif selected in ["MultiviewEqual", "TaskSpecificFusion"]:
    if selected == "TaskSpecificFusion":
        final_weight_path = RUN / "final_fusion_weights.json"
        weight_signature = hash_value({"run": run_signature, "view_oof": sha256_file(RUN / "predictions/view_oof.parquet")})
        if final_weight_path.exists():
            saved = json.loads(final_weight_path.read_text())
            assert saved["signature"] == weight_signature
            final_weights = np.array(saved["weights"])
        else:
            for target_index in range(2):
                final_weights[target_index], _ = choose_weights(oof_views[:, target_index, :], Y[:, target_index])
            atomic_json(final_weight_path, {"signature": weight_signature, "weights": final_weights.tolist(), "view_order": view_names})
    for view in view_names:
        final_pairs[view] = fit_pair(f"final_{view}_{best_single}", best_single, np.arange(len(X)), VIEWS[view])
        final_refit_seconds += final_pairs[view]["training_seconds"]
    final_recipe = {"kind": "three_view_fusion", "backbone": best_single, "view_order": view_names,
                    "weights": final_weights.tolist(), "weight_learning_data": "full-development cross-fitted view probabilities",
                    "views": {name: {"directory": str(pair["directory"]), "preprocessing": pair["preprocessing"], "classifiers": pair["statistics"]}
                              for name, pair in final_pairs.items()}}
else:
    final_pairs["all_features"] = fit_pair("final_" + selected, selected, np.arange(len(X)), BASE)
    final_refit_seconds = final_pairs["all_features"]["training_seconds"]
    final_recipe = {"kind": "single_model", "method": selected, "directory": str(final_pairs["all_features"]["directory"]),
                    "preprocessing": final_pairs["all_features"]["preprocessing"], "classifiers": final_pairs["all_features"]["statistics"]}

frozen = {"signature": run_signature, "selected_algorithm": selected, "selected_single_model": best_single,
          "training_records": 700000, "thresholds": thresholds, "recipe": final_recipe,
          "cv_summary_sha256": sha256_file(RUN / "cv_summary.csv"),
          "oof_sha256": sha256_file(RUN / "predictions/development_oof.parquet"),
          "locked_at": datetime.now(ZoneInfo("Asia/Shanghai")).isoformat(),
          "prior_supplemental_exploratory_evaluation": True, "config": config}
freeze_path = RUN / "frozen_protocol.json"
if freeze_path.exists():
    previous = json.loads(freeze_path.read_text())
    for name in ["signature", "selected_algorithm", "thresholds", "recipe", "cv_summary_sha256", "oof_sha256"]:
        assert previous[name] == frozen[name], name
    frozen = previous
else:
    atomic_json(freeze_path, frozen)
frozen_digest = sha256_file(freeze_path)
atomic_json(RUN / "frozen_protocol_receipt.json", {"path": str(freeze_path), "sha256": frozen_digest})
del X, oof, output_oof
gc.collect()

prediction_receipts = {}
for partition, filename, expected_rows in [("S", "supplemental_locked.parquet", 100000), ("K", "kaggle_test.parquet", 200000)]:
    path = RUN / "predictions" / ("holdout_probabilities.parquet" if partition == "S" else "kaggle_probabilities.parquet")
    if reuse_parent:
        source = PARENT / "predictions" / path.name
        shutil.copy2(source, path)
        score = pd.read_parquet(path)
        prediction_seconds = json.loads((PARENT / "prediction_receipts.json").read_text())[partition]["prediction_statistics"]["prediction_seconds"]
    else:
        source = processed / filename
        assert sha256_file(source) == release["artifact_sha256"][filename]
        frame = pd.read_parquet(source, columns=["encounter_id"] + BASE)
        identifiers = frame.encounter_id.astype(str).to_numpy()
        inputs = prepare_base_features(frame)
        if selected in ["MultiviewEqual", "TaskSpecificFusion"]:
            view_probability = np.empty((len(frame), 2, 3))
            prediction_seconds = 0.0
            for view_index, view in enumerate(view_names):
                view_probability[:, :, view_index], statistics = predict_pair(final_pairs[view], inputs, partition, identifiers)
                prediction_seconds += statistics["prediction_seconds"]
            started = time.perf_counter()
            probability = np.einsum("ntv,tv->nt", view_probability, final_weights)
            prediction_seconds += time.perf_counter() - started
        else:
            probability, statistics = predict_pair(final_pairs["all_features"], inputs, partition, identifiers)
            prediction_seconds = statistics["prediction_seconds"]
        score = pd.DataFrame({"encounter_id": identifiers, **{target: probability[:, index] for index, target in enumerate(targets)}})
        score.to_parquet(path, index=False)
        del frame, inputs, probability
        gc.collect()
    assert len(score) == expected_rows and not score.encounter_id.duplicated().any()
    assert not set(score.encounter_id) & set(IDS)
    assert np.isfinite(score[targets].to_numpy()).all()
    assert ((score[targets] >= 0) & (score[targets] <= 1)).all().all()
    prediction_receipts[partition] = {"path": str(path), "rows": len(score), "sha256": sha256_file(path),
                                       "frozen_protocol_sha256": frozen_digest,
                                       "prediction_statistics": {"prediction_seconds": prediction_seconds,
                                                                 "seconds_per_10000": prediction_seconds * 10000 / len(score)}}
    if partition == "K":
        sample = pd.read_csv(PROJECT_ROOT / "data/raw/sample_submission.csv", dtype={"encounter_id": str})
        submission = sample[["encounter_id"]].merge(score, on="encounter_id", how="left", validate="one_to_one")
        assert len(submission) == 200000 and not submission[targets].isna().any().any()
        if reuse_parent:
            shutil.copy2(PARENT / "submission.csv", RUN / "submission.csv")
        else:
            submission.to_csv(RUN / "submission.csv", index=False, float_format="%.12g")
        atomic_json(RUN / "submission_receipt.json", {"path": str(RUN / "submission.csv"), "rows": len(submission),
                    "columns": submission.columns.tolist(), "sha256": sha256_file(RUN / "submission.csv"),
                    "selected_algorithm": selected, "frozen_protocol_sha256": frozen_digest})
atomic_json(RUN / "prediction_receipts.json", prediction_receipts)

# Evaluation starts after the model recipe, thresholds and prediction files are frozen.
score = pd.read_parquet(prediction_receipts["S"]["path"])
truth = pd.read_parquet(processed / "supplemental_locked.parquet", columns=["encounter_id"] + targets)
truth = truth.set_index("encounter_id").loc[score.encounter_id]
S_Y = truth[targets].to_numpy(dtype=np.int8)
S_P = score[targets].to_numpy(dtype=float)
rows = []
for target_index, target in enumerate(targets):
    threshold = thresholds[target]["threshold"]
    tn, fp, fn, tp = confusion_matrix(S_Y[:, target_index], S_P[:, target_index] >= threshold, labels=[0, 1]).ravel()
    rows.append({"method": selected, "target": target, "records": len(truth), "events": int(S_Y[:, target_index].sum()),
                 "auc": roc_auc_score(S_Y[:, target_index], S_P[:, target_index]), "sensitivity": tp / (tp + fn),
                 "specificity": tn / (tn + fp), "threshold": threshold,
                 "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn)})


def auc_groups(probability):
    order = np.argsort(probability, kind="stable")
    sorted_probability = probability[order]
    sorted_groups = np.cumsum(np.r_[True, sorted_probability[1:] != sorted_probability[:-1]], dtype=np.int32) - 1
    groups = np.empty(len(probability), dtype=np.int32)
    groups[order] = sorted_groups
    return groups, int(sorted_groups[-1]) + 1


def weighted_auc(groups, count, labels, weights):
    positive = np.bincount(groups, weights=weights * labels, minlength=count)
    negative = np.bincount(groups, weights=weights * (1 - labels), minlength=count)
    return np.sum(positive * (np.cumsum(negative) - negative / 2)) / (positive.sum() * negative.sum())


bootstrap_path = RUN / "evaluation/auc_bootstrap.npz"
bootstrap_signature = {"frozen_protocol_sha256": frozen_digest, "prediction_sha256": prediction_receipts["S"]["sha256"], "seed": seed, "repetitions": 1000}
if bootstrap_path.exists():
    assert json.loads((RUN / "evaluation/auc_bootstrap_metadata.json").read_text())["signature"] == bootstrap_signature
    with np.load(bootstrap_path, allow_pickle=False) as archive:
        auc_samples = archive["auc"]
else:
    groups = [auc_groups(S_P[:, index]) for index in range(2)]
    auc_samples = np.empty((1000, 2))
    random = np.random.default_rng(seed)
    for repetition in range(1000):
        weights = np.bincount(random.integers(0, len(S_Y), size=len(S_Y)), minlength=len(S_Y)).astype(float)
        for index, (ranking, count) in enumerate(groups):
            auc_samples[repetition, index] = weighted_auc(ranking, count, S_Y[:, index], weights)
    np.savez_compressed(bootstrap_path, auc=auc_samples)
    atomic_json(RUN / "evaluation/auc_bootstrap_metadata.json", {"signature": bootstrap_signature, "conditional_on_fitted_models": True, "unit": "encounter"})
holdout = pd.DataFrame(rows)
holdout["auc_lower"], holdout["auc_upper"] = np.quantile(auc_samples, [0.025, 0.975], axis=0)
holdout.to_csv(RUN / "evaluation/holdout_metrics_with_intervals.csv", index=False)
receipt = json.loads((RUN / "submission_receipt.json").read_text())
parent_submission_record = json.loads((PARENT / "kaggle_submission.json").read_text())
if receipt["sha256"] == parent_submission_record["submission_sha256"]:
    kaggle = {**parent_submission_record, "submission_csv": receipt["path"], "selected_algorithm": selected,
              "identical_probability_file_reused": True, "extended_run_id": config["run_id"]}
else:
    kaggle = {"status": "awaiting_user_upload", "selected_algorithm": selected, "backbone": best_single,
              "submission_csv": receipt["path"], "submission_sha256": receipt["sha256"],
              "reference_user_reported_submission": parent_submission_record}
atomic_json(RUN / "kaggle_submission.json", kaggle)
result = {"status": "completed", "run_id": config["run_id"], "parent_run_id": config["parent_run_id"],
          "selected_algorithm": selected, "selected_single_model": best_single, "new_classifier_fits": len(fit_records),
          "new_cv_and_inner_classifier_fits": 140, "reused_parent_cv_classifier_fits": 50,
          "final_classifier_refit_seconds": final_refit_seconds, "reused_parent_final_model": reuse_parent,
          "holdout": holdout.to_dict(orient="records"), "holdout_macro_auc": float(holdout.auc.mean()),
          "holdout_macro_auc_interval": np.quantile(auc_samples.mean(axis=1), [0.025, 0.975]).tolist(),
          "submission_receipt": receipt, "kaggle_submission_status": kaggle["status"],
          "frozen_protocol_sha256": frozen_digest, "fit_warning_count": sum(len(row["warnings"]) for row in fit_records.values()),
          "prior_supplemental_exploratory_evaluation": True}
assert sha256_file(freeze_path) == frozen_digest
atomic_json(RUN / "summary.json", result)
atomic_json(RUN / "evaluation/summary.json", result)
atomic_json(RUN / "progress.json", result)
print(json.dumps(result, ensure_ascii=False, indent=2), flush=True)
